# CelebA CNN Embedding Training

Train a CNN on CelebA identities and use the normalized penultimate layer as the face embedding.

In [1]:
import torch
print(torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)


2.6.0+cu124
cuda available: True
cuda version: 12.4


In [2]:
from pathlib import Path
import random
import time
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
print("device:", device)

device: cuda


In [3]:
DATASET_ROOT = Path(r"C:\DSP\img_align_celeba")
IMAGE_DIR = DATASET_ROOT / "img_align_celeba"
IDENTITY_FILE = DATASET_ROOT / "Anno" / "identity_CelebA.txt"
CHECKPOINT_DIR = Path(r"C:\DSP\checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
EMBED_DIM = 256
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
TRAIN_SPLIT_RATIO = 0.9
SEED = 42

MAX_IDENTITIES = None
MAX_IMAGES_PER_IDENTITY = None
USE_PRETRAINED = False
NUM_WORKERS = 0
PIN_MEMORY = device.type == "cuda"

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert IMAGE_DIR.exists(), IMAGE_DIR
assert IDENTITY_FILE.exists(), IDENTITY_FILE
print("images:", IMAGE_DIR)
print("identities:", IDENTITY_FILE)

images: C:\DSP\img_align_celeba\img_align_celeba
identities: C:\DSP\img_align_celeba\Anno\identity_CelebA.txt


In [4]:
def load_identity_records(identity_file, image_dir, max_identities=None, max_images_per_identity=None):
    grouped = defaultdict(list)
    with open(identity_file, "r", encoding="utf-8") as f:
        for line in f:
            image_name, identity = line.strip().split()
            image_path = image_dir / image_name
            if image_path.exists():
                grouped[int(identity)].append(image_path)

    identity_ids = sorted(grouped)
    if max_identities is not None:
        identity_ids = identity_ids[:max_identities]

    records = []
    reindexed = {identity_id: idx for idx, identity_id in enumerate(identity_ids)}
    for identity_id in identity_ids:
        image_paths = grouped[identity_id]
        if max_images_per_identity is not None:
            image_paths = image_paths[:max_images_per_identity]
        for image_path in image_paths:
            records.append((image_path, reindexed[identity_id], identity_id))
    return records, reindexed


def split_within_identity(records, train_ratio=0.9, seed=42):
    rng = random.Random(seed)
    grouped = defaultdict(list)
    for image_path, class_idx, original_identity in records:
        grouped[class_idx].append((image_path, class_idx, original_identity))

    train_records = []
    val_records = []
    for _, items in grouped.items():
        items = items[:]
        rng.shuffle(items)

        if len(items) == 1:
            train_records.extend(items)
            continue

        cutoff = int(len(items) * train_ratio)
        cutoff = min(max(cutoff, 1), len(items) - 1)
        train_records.extend(items[:cutoff])
        val_records.extend(items[cutoff:])

    return train_records, val_records


class CelebAIdentityDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        image_path, class_idx, original_identity = self.records[idx]
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)
        return image, class_idx, str(image_path), original_identity


train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

records, id_map = load_identity_records(
    IDENTITY_FILE,
    IMAGE_DIR,
    max_identities=MAX_IDENTITIES,
    max_images_per_identity=MAX_IMAGES_PER_IDENTITY,
)
train_records, val_records = split_within_identity(records, train_ratio=TRAIN_SPLIT_RATIO, seed=SEED)

train_dataset = CelebAIdentityDataset(train_records, train_transform)
val_dataset = CelebAIdentityDataset(val_records, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print("num classes:", len(id_map))
print("train images:", len(train_dataset))
print("val images:", len(val_dataset))

num classes: 10177
train images: 178978
val images: 23621


In [5]:
class FaceEmbeddingCNN(nn.Module):
    def __init__(self, embedding_dim, num_classes):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if USE_PRETRAINED else None
        backbone = models.resnet18(weights=weights)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.embedding = nn.Sequential(
            nn.Linear(in_features, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.embedding(features)
        normalized_embedding = nn.functional.normalize(embedding, p=2, dim=1)
        logits = self.classifier(embedding)
        return normalized_embedding, logits


model = FaceEmbeddingCNN(embedding_dim=EMBED_DIM, num_classes=len(id_map)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
model

FaceEmbeddingCNN(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True,

In [6]:
smoke_start = time.time()
smoke_images, smoke_labels, _, _ = next(iter(train_loader))
print("first batch shape:", tuple(smoke_images.shape), tuple(smoke_labels.shape))
print("first batch load time:", f"{time.time() - smoke_start:.2f}s")

first batch shape: (32, 3, 224, 224) (32,)
first batch load time: 0.40s


In [7]:
def format_seconds(seconds):
    seconds = max(0, int(seconds))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours > 0:
        return f"{hours:d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def run_epoch(model, loader, optimizer=None, epoch_idx=None, total_epochs=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    epoch_start = time.time()
    mode = "train" if is_train else "val"

    progress = tqdm(loader, desc=f"epoch {epoch_idx}/{total_epochs} [{mode}]", leave=False, dynamic_ncols=True)
    for batch_idx, (images, labels, _, _) in enumerate(progress, start=1):
        batch_start = time.time()
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            _, logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size

        elapsed = time.time() - epoch_start
        batches_done = batch_idx
        batches_left = len(loader) - batches_done
        avg_batch_time = elapsed / batches_done
        eta = batches_left * avg_batch_time
        speed = total_examples / max(elapsed, 1e-6)

        progress.set_postfix({
            "loss": f"{total_loss / total_examples:.4f}",
            "acc": f"{total_correct / total_examples:.4f}",
            "img_s": f"{speed:.1f}",
            "eta": format_seconds(eta),
        })

    epoch_time = time.time() - epoch_start
    return total_loss / total_examples, total_correct / total_examples, epoch_time


best_val_loss = float("inf")
history = []
training_start = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, train_time = run_epoch(model, train_loader, optimizer=optimizer, epoch_idx=epoch, total_epochs=EPOCHS)
    val_loss, val_acc, val_time = run_epoch(model, val_loader, optimizer=None, epoch_idx=epoch, total_epochs=EPOCHS)
    scheduler.step()

    epoch_time = train_time + val_time
    elapsed_total = time.time() - training_start
    avg_epoch_time = elapsed_total / epoch
    remaining_epochs = EPOCHS - epoch
    training_eta = remaining_epochs * avg_epoch_time
    train_speed = len(train_dataset) / max(train_time, 1e-6)
    val_speed = len(val_dataset) / max(val_time, 1e-6)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "epoch_time_sec": epoch_time,
    })

    print(
        f"epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_speed={train_speed:.1f} img/s | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_speed={val_speed:.1f} img/s | "
        f"epoch_time={format_seconds(epoch_time)} | total_eta={format_seconds(training_eta)}"
    )

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": history,
        "embedding_dim": EMBED_DIM,
        "num_classes": len(id_map),
        "image_size": IMAGE_SIZE,
        "id_map": id_map,
    }

    latest_path = CHECKPOINT_DIR / "celeba_embedding_latest.pt"
    torch.save(checkpoint, latest_path)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_path = CHECKPOINT_DIR / "celeba_embedding_best.pt"
        torch.save(checkpoint, best_path)

print("saved to:", CHECKPOINT_DIR)

epoch 1/10 [train]:   0%|          | 0/5594 [00:00<?, ?it/s]

epoch 1/10 [val]:   0%|          | 0/739 [00:00<?, ?it/s]

epoch 01/10 | train_loss=9.1713 train_acc=0.0001 train_speed=106.8 img/s | val_loss=9.1012 val_acc=0.0002 val_speed=271.9 img/s | epoch_time=29:22 | total_eta=4:24:18


epoch 2/10 [train]:   0%|          | 0/5594 [00:00<?, ?it/s]

epoch 2/10 [val]:   0%|          | 0/739 [00:00<?, ?it/s]

...

epoch 9/10 [val]:   0%|          | 0/739 [00:00<?, ?it/s]

epoch 09/10 | train_loss=4.8820 train_acc=0.1414 train_speed=128.4 img/s | val_loss=5.7114 val_acc=0.1135 val_speed=341.1 img/s | epoch_time=24:22 | total_eta=25:09


epoch 10/10 [train]:   0%|          | 0/5594 [00:00<?, ?it/s]

epoch 10/10 [val]:   0%|          | 0/739 [00:00<?, ?it/s]

epoch 10/10 | train_loss=4.7332 train_acc=0.1566 train_speed=128.4 img/s | val_loss=5.6100 val_acc=0.1194 val_speed=338.9 img/s | epoch_time=24:23 | total_eta=00:00
saved to: C:\DSP\checkpoints


In [8]:
@torch.no_grad()
def extract_embeddings(model, loader, output_path):
    model.eval()
    rows = []
    progress = tqdm(loader, desc="extract embeddings", dynamic_ncols=True)
    for images, labels, paths, original_ids in progress:
        images = images.to(device, non_blocking=True)
        embeddings, _ = model(images)
        embeddings = embeddings.cpu()
        labels = labels.cpu()
        for idx in range(embeddings.size(0)):
            rows.append({
                "path": paths[idx],
                "class_idx": int(labels[idx]),
                "original_identity": int(original_ids[idx]),
                "embedding": embeddings[idx],
            })
        progress.set_postfix({"saved": len(rows)})
    torch.save(rows, output_path)
    return output_path


embedding_dump = CHECKPOINT_DIR / "celeba_val_embeddings.pt"
extract_embeddings(model, val_loader, embedding_dump)
print("embedding file:", embedding_dump)

extract embeddings:   0%|          | 0/739 [00:00<?, ?it/s]

embedding file: C:\DSP\checkpoints\celeba_val_embeddings.pt
